# Hamiltonian Simulation — The Transverse-Field Ising Chain

**Lecture Exercises — Quantum Computing**

Simulating the dynamics of quantum many-body systems is one of the most natural and compelling applications of quantum computers — indeed, it was Feynman’s original motivation for proposing them. In this notebook we study:

1. **Quantum time evolution** and why it is hard classically
2. **Product formulas (Trotterisation)** — the Lie–Trotter–Suzuki decomposition
3. **The transverse-field Ising model** as a concrete lattice Hamiltonian
4. **Building Trotter circuits** from elementary gates
5. **Measuring observables** during time evolution (magnetisation, correlations)
6. **Trotter error analysis** — convergence with step size
7. **Higher-order product formulas** (second-order Suzuki–Trotter)


## 1. Setup and Imports

In [ ]:
# If running locally, uncomment to install:
# !pip install qiskit qiskit-aer scipy

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, SparsePauliOp, Operator

simulator = AerSimulator()

## 2. Quantum Time Evolution

### The Schrödinger equation

The state of an isolated quantum system evolves according to:

$$i\hbar \frac{\partial}{\partial t} |\psi(t)\rangle = H |\psi(t)\rangle$$

For a time-independent Hamiltonian $H$, the solution is (setting $\hbar = 1$):

$$|\psi(t)\rangle = e^{-iHt} |\psi(0)\rangle$$

The time-evolution operator $U(t) = e^{-iHt}$ is **unitary** (since $H$ is Hermitian), so it can in principle be implemented as a quantum circuit.

### Why is this hard classically?

For a system of $n$ qubits, $H$ is a $2^n \times 2^n$ matrix. Computing the matrix exponential $e^{-iHt}$ by brute-force diagonalisation costs $O(2^{3n})$ — exponentially expensive. Even storing the full state vector requires $O(2^n)$ memory.

A quantum computer with $n$ qubits can represent $|\psi(t)\rangle$ natively. The key algorithmic challenge is decomposing $e^{-iHt}$ into a sequence of elementary quantum gates **efficiently** (polynomial in $n$ and $t$).

### Exact diagonalisation (small systems)

For small systems we can still compute the exact evolution classically as a reference:

In [ ]:
def exact_time_evolution(H_matrix, psi0, t):
    """Compute exact time evolution |psi(t)> = exp(-iHt)|psi(0)>.
    
    Parameters:
        H_matrix: Hamiltonian as a numpy array
        psi0: initial state as a numpy array
        t: evolution time
    
    Returns:
        numpy array: the time-evolved state
    """
    U = expm(-1j * t * H_matrix)
    return U @ psi0


# Example: single qubit in a magnetic field H = -h*X
h = 1.0
H_1q = -h * np.array([[0, 1], [1, 0]])  # -h * X

psi0 = np.array([1, 0], dtype=complex)  # |0>
t = np.pi / 2

psi_t = exact_time_evolution(H_1q, psi0, t)
print(f"H = -{h}*X")
print(f"Initial state: |0>")
print(f"Time: t = pi/2")
print(f"Final state: {psi_t.round(6)}")
print(f"|psi(t)|^2: |<0|psi>|^2 = {abs(psi_t[0])**2:.4f}, |<1|psi>|^2 = {abs(psi_t[1])**2:.4f}")
print(f"\nThis is |1> (up to a phase), showing a full spin flip under X.")

## 3. Product Formulas (Trotterisation)

### The problem

Most physically interesting Hamiltonians decompose as a sum of simpler terms:

$$H = \sum_{k=1}^{L} H_k$$

where each $H_k$ acts on a few qubits and can be exponentiated efficiently. However, in general:

$$e^{-i(A + B)t} \neq e^{-iAt} e^{-iBt}$$

unless $[A, B] = 0$. The non-commutativity of quantum operators is what makes simulation non-trivial.

### First-order Lie–Trotter formula

The **first-order product formula** approximates:

$$e^{-iHt} = e^{-i(H_1 + H_2 + \cdots + H_L)t} \approx \left( \prod_{k=1}^{L} e^{-iH_k t/r} \right)^r$$

where $r$ is the number of **Trotter steps**. The error per step is $O(t^2/r^2)$, giving a total error of $O(t^2/r)$. More precisely, by the Baker–Campbell–Hausdorff formula:

$$e^{-iA\delta t} e^{-iB\delta t} = e^{-i(A+B)\delta t - \frac{1}{2}[A,B]\delta t^2 + \cdots}$$

so the leading error is proportional to $[H_k, H_j]$.

### Second-order Suzuki–Trotter formula

A better approximation is the **symmetric (second-order) Trotter formula**:

$$e^{-iHt} \approx \left( \prod_{k=1}^{L} e^{-iH_k t/(2r)} \cdot \prod_{k=L}^{1} e^{-iH_k t/(2r)} \right)^r$$

This has error $O(t^3/r^2)$ — one order better than the first-order formula.

Let’s see Trotterisation in action on a simple 2-operator example before moving to the Ising chain.

In [ ]:
# Demonstrate Trotter error for e^{-i(A+B)t} where A = Z, B = X
A = np.array([[1, 0], [0, -1]], dtype=complex)  # Z
B = np.array([[0, 1], [1, 0]], dtype=complex)    # X
H_demo = A + B

t = 1.0
U_exact = expm(-1j * t * H_demo)

print("Trotter error for H = Z + X, t = 1.0\n")
print(f"{'r (steps)':>10s}  {'1st order err':>14s}  {'2nd order err':>14s}")
print("-" * 44)

for r in [1, 2, 5, 10, 20, 50, 100]:
    dt = t / r
    
    # First-order Trotter: (e^{-iA*dt} e^{-iB*dt})^r
    U_A = expm(-1j * dt * A)
    U_B = expm(-1j * dt * B)
    U_trotter1 = np.linalg.matrix_power(U_B @ U_A, r)
    err1 = np.linalg.norm(U_trotter1 - U_exact)
    
    # Second-order Trotter: (e^{-iA*dt/2} e^{-iB*dt} e^{-iA*dt/2})^r
    U_A_half = expm(-1j * dt / 2 * A)
    U_trotter2 = np.linalg.matrix_power(U_A_half @ U_B @ U_A_half, r)
    err2 = np.linalg.norm(U_trotter2 - U_exact)
    
    print(f"{r:10d}  {err1:14.6e}  {err2:14.6e}")

print("\n1st order: error ~ 1/r.  2nd order: error ~ 1/r^2.")

In [ ]:
# Visualise the scaling
r_values = np.arange(1, 101)
errors_1st = []
errors_2nd = []

for r in r_values:
    dt = t / r
    U_A = expm(-1j * dt * A)
    U_B = expm(-1j * dt * B)
    U_A_half = expm(-1j * dt / 2 * A)
    
    U_t1 = np.linalg.matrix_power(U_B @ U_A, r)
    U_t2 = np.linalg.matrix_power(U_A_half @ U_B @ U_A_half, r)
    
    errors_1st.append(np.linalg.norm(U_t1 - U_exact))
    errors_2nd.append(np.linalg.norm(U_t2 - U_exact))

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(r_values, errors_1st, 'b-', linewidth=2, label='1st order Trotter')
ax.loglog(r_values, errors_2nd, 'r-', linewidth=2, label='2nd order Suzuki-Trotter')
ax.loglog(r_values, 2.0 / r_values, 'b--', alpha=0.4, label='$\sim 1/r$')
ax.loglog(r_values, 2.0 / r_values**2, 'r--', alpha=0.4, label='$\sim 1/r^2$')
ax.set_xlabel('Number of Trotter steps $r$', fontsize=12)
ax.set_ylabel('Operator norm error $\|U_{\mathrm{Trotter}} - U_{\mathrm{exact}}\|$', fontsize=12)
ax.set_title('Trotter Error Scaling', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 4. The Transverse-Field Ising Model

### The model

The **transverse-field Ising model (TFIM)** is one of the most important models in condensed matter physics. It describes a chain of spin-$1/2$ particles with nearest-neighbour interactions and a transverse magnetic field:

$$H = -J \sum_{i=0}^{n-2} Z_i Z_{i+1} \;-\; h \sum_{i=0}^{n-1} X_i$$

where:
- $J$ is the **coupling strength** (Ising interaction between neighbours)
- $h$ is the **transverse field strength**
- $Z_i, X_i$ are Pauli operators on site $i$

### Physics

- When $h = 0$: the ground states are the fully aligned states $|00\ldots0\rangle$ and $|11\ldots1\rangle$ (ferromagnetic phase).
- When $J = 0$: the ground state is $|+\rangle^{\otimes n}$ (paramagnetic phase, all spins aligned with the field).
- The model has a **quantum phase transition** at $J = h$ in the thermodynamic limit ($n \to \infty$).

### Decomposition for Trotterisation

The Hamiltonian splits naturally into two non-commuting parts:

$$H = H_{ZZ} + H_X$$

where $H_{ZZ} = -J \sum_i Z_i Z_{i+1}$ and $H_X = -h \sum_i X_i$.

The key insight is that **each part** can be exponentiated efficiently:
- $e^{-iH_X t} = \prod_i e^{ihX_i t} = \prod_i R_x(-2ht)$, since the $X_i$ terms commute with each other.
- $e^{-iH_{ZZ} t} = \prod_i e^{iJ Z_i Z_{i+1} t}$, since the $Z_i Z_{i+1}$ terms commute with each other (all diagonal in the $Z$ basis).

In [ ]:
def tfim_hamiltonian(n, J=1.0, h=1.0):
    """Build the TFIM Hamiltonian as a SparsePauliOp.
    
    H = -J * sum_i Z_i Z_{i+1} - h * sum_i X_i
    
    Parameters:
        n: number of qubits (spins)
        J: coupling strength
        h: transverse field strength
    
    Returns:
        SparsePauliOp
    """
    terms = []
    
    # ZZ interactions: -J * Z_i Z_{i+1}
    for i in range(n - 1):
        pauli = ['I'] * n
        pauli[i] = 'Z'
        pauli[i + 1] = 'Z'
        # SparsePauliOp: qubit 0 is rightmost
        terms.append((''.join(reversed(pauli)), -J))
    
    # Transverse field: -h * X_i
    for i in range(n):
        pauli = ['I'] * n
        pauli[i] = 'X'
        terms.append((''.join(reversed(pauli)), -h))
    
    return SparsePauliOp.from_list(terms)


# Build the TFIM for n=4
n = 4
J, h = 1.0, 0.5
H_tfim = tfim_hamiltonian(n, J, h)
print(f"TFIM Hamiltonian (n={n}, J={J}, h={h}):")
print(H_tfim)

# Exact spectrum
H_mat = H_tfim.to_matrix()
eigenvalues = np.linalg.eigvalsh(H_mat)
print(f"\nEnergy spectrum: {eigenvalues.real.round(4)}")
print(f"Ground state energy: {min(eigenvalues.real):.6f}")
print(f"Energy gap: {eigenvalues.real[1] - eigenvalues.real[0]:.6f}")

In [ ]:
# Visualise the energy spectrum as a function of h/J
n = 4
J = 1.0
h_values = np.linspace(0, 3, 80)

fig, ax = plt.subplots(figsize=(10, 6))

all_spectra = []
for h_val in h_values:
    H = tfim_hamiltonian(n, J, h_val)
    evals = np.sort(np.linalg.eigvalsh(H.to_matrix()).real)
    all_spectra.append(evals)

all_spectra = np.array(all_spectra)

# Plot lowest few energy levels
for level in range(min(8, 2**n)):
    ax.plot(h_values, all_spectra[:, level], linewidth=1.5, alpha=0.7)

ax.axvline(x=J, color='gray', linestyle='--', alpha=0.5, label='$h = J$')
ax.set_xlabel('Transverse field $h/J$', fontsize=12)
ax.set_ylabel('Energy', fontsize=12)
ax.set_title(f'TFIM Energy Spectrum ($n = {n}$)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The energy gap closes near h/J = 1, signalling the quantum phase transition.")

## 5. Building Trotter Circuits

### Elementary gates for the TFIM

We need two types of building blocks:

**1. The $ZZ$ interaction: $e^{iJ \delta t\, Z_i Z_{i+1}}$**

Since $Z_i Z_{i+1}$ has eigenvalues $\pm 1$, this is a conditional phase rotation. The standard decomposition is:

$$e^{i\theta Z_i Z_{i+1}} = \text{CNOT}_{i,i+1} \cdot R_z(2\theta)_{i+1} \cdot \text{CNOT}_{i,i+1}$$

For the TFIM, $\theta = J\delta t$.

**2. The transverse field: $e^{ih \delta t\, X_i}$**

This is simply a rotation around the $x$-axis:

$$e^{ih\delta t\, X_i} = R_x(-2h\delta t)_i$$

In [ ]:
def trotter_step_1st_order(n, J, h, dt):
    """Build a single first-order Trotter step for the TFIM.
    
    U_1(dt) = exp(-i H_X dt) * exp(-i H_ZZ dt)
            = prod_i Rx(2*h*dt, i) * prod_i [CNOT, Rz(-2*J*dt), CNOT]
    
    Parameters:
        n: number of qubits
        J: coupling strength
        h: transverse field
        dt: time step
    
    Returns:
        QuantumCircuit
    """
    qc = QuantumCircuit(n, name=f'Trotter1({dt:.3f})')
    
    # ZZ interactions: exp(i*J*dt * Z_i Z_{i+1}) for each bond
    for i in range(n - 1):
        qc.cx(i, i + 1)
        qc.rz(-2 * J * dt, i + 1)  # angle = -2*J*dt (the minus from H = -J*ZZ)
        qc.cx(i, i + 1)
    
    # Transverse field: exp(i*h*dt * X_i) = Rx(-2*h*dt) for each site
    for i in range(n):
        qc.rx(-2 * h * dt, i)  # Rx(-2*h*dt) = exp(i*h*dt*X)
    
    return qc


# Build and draw a single Trotter step
step = trotter_step_1st_order(4, J=1.0, h=0.5, dt=0.1)
step.draw('mpl')

In [ ]:
def trotter_step_2nd_order(n, J, h, dt):
    """Build a single second-order (symmetric) Trotter step for the TFIM.
    
    U_2(dt) = exp(-i H_X dt/2) * exp(-i H_ZZ dt) * exp(-i H_X dt/2)
    
    The symmetric splitting cancels the leading-order error.
    """
    qc = QuantumCircuit(n, name=f'Trotter2({dt:.3f})')
    
    # Half-step of transverse field
    for i in range(n):
        qc.rx(-2 * h * dt / 2, i)
    
    # Full step of ZZ interactions
    for i in range(n - 1):
        qc.cx(i, i + 1)
        qc.rz(-2 * J * dt, i + 1)
        qc.cx(i, i + 1)
    
    # Half-step of transverse field
    for i in range(n):
        qc.rx(-2 * h * dt / 2, i)
    
    return qc


step2 = trotter_step_2nd_order(4, J=1.0, h=0.5, dt=0.1)
step2.draw('mpl')

In [ ]:
def trotter_circuit(n, J, h, total_time, n_steps, order=1):
    """Build the full Trotter circuit for TFIM time evolution.
    
    Parameters:
        n: number of qubits
        J, h: Hamiltonian parameters
        total_time: total evolution time T
        n_steps: number of Trotter steps r
        order: 1 for first-order, 2 for second-order
    
    Returns:
        QuantumCircuit
    """
    dt = total_time / n_steps
    qc = QuantumCircuit(n)
    
    for step in range(n_steps):
        if order == 1:
            step_qc = trotter_step_1st_order(n, J, h, dt)
        else:
            step_qc = trotter_step_2nd_order(n, J, h, dt)
        qc.compose(step_qc, inplace=True)
        qc.barrier()
    
    return qc


# Example: full circuit for T=1.0, r=5 steps
qc_full = trotter_circuit(4, J=1.0, h=0.5, total_time=1.0, n_steps=5, order=1)
print(f"Full Trotter circuit: {qc_full.num_qubits} qubits, depth ~ {qc_full.depth()}")
qc_full.draw('mpl', fold=-1)

## 6. Comparing Trotter Evolution with Exact Dynamics

Now let’s verify our Trotter circuits by comparing the time-evolved state against the exact solution from matrix exponentiation.

In [ ]:
def fidelity(psi1, psi2):
    """Compute the state fidelity |<psi1|psi2>|^2."""
    return abs(np.vdot(psi1, psi2))**2


def trotter_evolve(n, J, h, total_time, n_steps, order=1, initial_state=None):
    """Evolve an initial state using the Trotter circuit.
    
    Parameters:
        n: number of qubits
        J, h: Hamiltonian parameters
        total_time: total evolution time
        n_steps: number of Trotter steps
        order: Trotter order (1 or 2)
        initial_state: Statevector (default: |00...0>)
    
    Returns:
        Statevector: the time-evolved state
    """
    if initial_state is None:
        initial_state = Statevector.from_label('0' * n)
    
    qc = trotter_circuit(n, J, h, total_time, n_steps, order)
    return initial_state.evolve(qc)


# Compare Trotter vs exact for n=4
n = 4
J, h = 1.0, 0.5
T = 2.0

H_mat = tfim_hamiltonian(n, J, h).to_matrix()
psi0 = np.zeros(2**n, dtype=complex)
psi0[0] = 1.0  # |0000>

# Exact evolution
psi_exact = exact_time_evolution(H_mat, psi0, T)

print(f"TFIM: n={n}, J={J}, h={h}, T={T}")
print(f"Initial state: |{'0'*n}>\n")
print(f"{'Steps r':>8s}  {'1st order F':>12s}  {'2nd order F':>12s}  {'1st err':>10s}  {'2nd err':>10s}")
print("-" * 60)

for r in [1, 2, 5, 10, 20, 50]:
    # First order
    sv1 = trotter_evolve(n, J, h, T, r, order=1)
    F1 = fidelity(psi_exact, np.array(sv1))
    
    # Second order
    sv2 = trotter_evolve(n, J, h, T, r, order=2)
    F2 = fidelity(psi_exact, np.array(sv2))
    
    print(f"{r:8d}  {F1:12.8f}  {F2:12.8f}  {1-F1:10.2e}  {1-F2:10.2e}")

In [ ]:
# Visualise fidelity vs Trotter steps
r_values = list(range(1, 51))
fidelities_1st = []
fidelities_2nd = []

for r in r_values:
    sv1 = trotter_evolve(n, J, h, T, r, order=1)
    sv2 = trotter_evolve(n, J, h, T, r, order=2)
    fidelities_1st.append(fidelity(psi_exact, np.array(sv1)))
    fidelities_2nd.append(fidelity(psi_exact, np.array(sv2)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1.plot(r_values, fidelities_1st, 'b-o', markersize=3, label='1st order')
ax1.plot(r_values, fidelities_2nd, 'r-s', markersize=3, label='2nd order')
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
ax1.set_xlabel(f'Trotter steps $r$', fontsize=12)
ax1.set_ylabel(r'Fidelity $|\langle\psi_{\mathrm{exact}}|\psi_{\mathrm{Trotter}}\rangle|^2$', fontsize=12)
ax1.set_title(f'Fidelity vs Trotter Steps', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Log scale for infidelity
infid_1st = [1 - f for f in fidelities_1st]
infid_2nd = [1 - f for f in fidelities_2nd]
ax2.loglog(r_values, infid_1st, 'b-o', markersize=3, label='1st order')
ax2.loglog(r_values, infid_2nd, 'r-s', markersize=3, label='2nd order')
ax2.loglog(r_values, [3.0/r**2 for r in r_values], 'b--', alpha=0.4, label='$\sim 1/r^2$')
ax2.loglog(r_values, [3.0/r**4 for r in r_values], 'r--', alpha=0.4, label='$\sim 1/r^4$')
ax2.set_xlabel('Trotter steps $r$', fontsize=12)
ax2.set_ylabel('Infidelity $1 - F$', fontsize=12)
ax2.set_title('Trotter Error Scaling', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("The infidelity scales as 1/r^2 (1st order) and 1/r^4 (2nd order).")
print("This is consistent with the operator error scaling of 1/r and 1/r^2 respectively,")
print("since fidelity error ~ (operator error)^2.")

## 7. Measuring Observables During Time Evolution

In physics, we are usually interested in how **observables** (expectation values of operators) evolve over time. Key observables for the Ising chain include:

- **Magnetisation along $z$**: $\langle M_z \rangle = \frac{1}{n} \sum_i \langle Z_i \rangle$
- **Magnetisation along $x$**: $\langle M_x \rangle = \frac{1}{n} \sum_i \langle X_i \rangle$
- **Nearest-neighbour correlations**: $\langle Z_i Z_{i+1} \rangle$
- **Entanglement entropy** of a bipartition

Let’s track these through the time evolution of the TFIM.

In [ ]:
def measure_observables(statevector, n):
    """Compute physical observables from a statevector.
    
    Returns:
        dict with Mz, Mx, ZZ correlations, and individual <Zi>, <Xi>
    """
    sv = statevector if isinstance(statevector, Statevector) else Statevector(statevector)
    
    # Average magnetisations
    Mz = 0
    Mx = 0
    Zi_list = []
    Xi_list = []
    for i in range(n):
        pauli_z = ['I'] * n
        pauli_z[i] = 'Z'
        op_z = SparsePauliOp.from_list([(''.join(reversed(pauli_z)), 1.0)])
        zi = sv.expectation_value(op_z).real
        Zi_list.append(zi)
        Mz += zi
        
        pauli_x = ['I'] * n
        pauli_x[i] = 'X'
        op_x = SparsePauliOp.from_list([(''.join(reversed(pauli_x)), 1.0)])
        xi = sv.expectation_value(op_x).real
        Xi_list.append(xi)
        Mx += xi
    
    Mz /= n
    Mx /= n
    
    # ZZ correlations
    ZZ_list = []
    for i in range(n - 1):
        pauli_zz = ['I'] * n
        pauli_zz[i] = 'Z'
        pauli_zz[i + 1] = 'Z'
        op_zz = SparsePauliOp.from_list([(''.join(reversed(pauli_zz)), 1.0)])
        ZZ_list.append(sv.expectation_value(op_zz).real)
    
    return {
        'Mz': Mz, 'Mx': Mx,
        'Zi': Zi_list, 'Xi': Xi_list,
        'ZZ': ZZ_list,
    }


# Test on the initial state |0000>
n = 4
sv0 = Statevector.from_label('0' * n)
obs0 = measure_observables(sv0, n)
print(f"Observables for |{'0'*n}>:")
print(f"  <Mz> = {obs0['Mz']:.4f}  (all spins up -> +1)")
print(f"  <Mx> = {obs0['Mx']:.4f}  (no x-polarisation)")
print(f"  <Z_i>: {[f'{z:.2f}' for z in obs0['Zi']]}")
print(f"  <Z_i Z_{{i+1}}>: {[f'{zz:.2f}' for zz in obs0['ZZ']]}")

In [ ]:
# Time evolution of observables
n = 4
J, h = 1.0, 1.0  # At the critical point!
H_mat = tfim_hamiltonian(n, J, h).to_matrix()

psi0 = np.zeros(2**n, dtype=complex)
psi0[0] = 1.0  # |0000>: fully polarised along z

# Evolve in small time steps using exact diagonalisation
n_times = 200
T_max = 6.0
times = np.linspace(0, T_max, n_times)

Mz_exact = []
Mx_exact = []
ZZ_avg_exact = []

for t in times:
    psi_t = exact_time_evolution(H_mat, psi0, t)
    obs = measure_observables(psi_t, n)
    Mz_exact.append(obs['Mz'])
    Mx_exact.append(obs['Mx'])
    ZZ_avg_exact.append(np.mean(obs['ZZ']))

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(times, Mz_exact, 'b-', linewidth=2)
axes[0].set_ylabel(r'$\langle M_z \rangle$', fontsize=12)
axes[0].set_title(r'TFIM Dynamics: $n={n}$, $J={J}$, $h={h}$ (critical point), initial state $|0000\rangle$', fontsize=13)
axes[0].grid(True, alpha=0.3)

axes[1].plot(times, Mx_exact, 'r-', linewidth=2)
axes[1].set_ylabel(r'$\langle M_x \rangle$', fontsize=12)
axes[1].grid(True, alpha=0.3)

axes[2].plot(times, ZZ_avg_exact, 'g-', linewidth=2)
axes[2].set_ylabel(r'$\langle Z_i Z_{i+1} \rangle_{\mathrm{avg}}$', fontsize=12)
axes[2].set_xlabel(r'Time $t$', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Starting from |0000> (fully z-polarised), the transverse field causes")
print("the z-magnetisation to oscillate as spins precess. At the critical point")
print("h=J, the dynamics are particularly rich with quasi-periodic revivals.")

## 8. Trotter vs exact dynamics of observables

Let’s compare the Trotter-simulated dynamics of $\langle M_z(t) \rangle$ against the exact result for different Trotter step counts.

In [ ]:
n = 4
J, h = 1.0, 1.0
T_max = 4.0
n_times = 100
times = np.linspace(0, T_max, n_times)

H_mat = tfim_hamiltonian(n, J, h).to_matrix()
psi0_np = np.zeros(2**n, dtype=complex)
psi0_np[0] = 1.0

# Exact
Mz_exact = []
for t in times:
    psi_t = exact_time_evolution(H_mat, psi0_np, t)
    obs = measure_observables(psi_t, n)
    Mz_exact.append(obs['Mz'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, r_per_unit_time in enumerate([1, 2, 5, 20]):
    ax = axes[idx]
    ax.plot(times, Mz_exact, 'k-', linewidth=2, alpha=0.5, label='Exact')
    
    for order, color, ls in [(1, 'blue', '-'), (2, 'red', '-')]:
        Mz_trotter = []
        for t in times:
            if t == 0:
                Mz_trotter.append(1.0)
                continue
            r = max(1, int(np.ceil(r_per_unit_time * t)))
            sv_t = trotter_evolve(n, J, h, t, r, order=order)
            obs = measure_observables(sv_t, n)
            Mz_trotter.append(obs['Mz'])
        label = f'Order {order}'
        ax.plot(times, Mz_trotter, color=color, linestyle=ls, linewidth=1.5, 
                alpha=0.8, label=label)
    
    ax.set_title(f'$r/t = {r_per_unit_time}$ steps per unit time', fontsize=12)
    ax.set_xlabel(r'Time $t$', fontsize=11)
    ax.set_ylabel(r'$\langle M_z \rangle$', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Trotter vs Exact Magnetisation Dynamics ($n={n}$, $J=h={J}$)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

With just 1–2 steps per unit time, the first-order Trotter formula already captures the qualitative behaviour but drifts over long times. The second-order formula is significantly more accurate at the same 2-qubit gate cost. With 5–20 steps per unit time, both formulas give excellent quantitative agreement.

## 9. Quench Dynamics: Different Initial States

A **quantum quench** is a sudden change in the Hamiltonian. The system is prepared in the ground state of one Hamiltonian and then evolved under a different one. Let’s explore quench dynamics for several initial states.

In [ ]:
n = 4
J, h = 1.0, 1.0
T_max = 6.0
n_times = 150
times = np.linspace(0, T_max, n_times)
H_mat = tfim_hamiltonian(n, J, h).to_matrix()

# Different initial states
initial_states = {
    '|0000> (all up)': Statevector.from_label('0' * n),
    '|1010> (Antiferromagnetic state)': Statevector.from_label('1010'),
    '|++++> (x-polarised)': Statevector.from_label('+' * n),
    '|1000> (domain wall)': Statevector.from_label('1000'),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (label, sv0) in enumerate(initial_states.items()):
    psi0_np = np.array(sv0)
    
    Mz_t = []
    Mx_t = []
    for t in times:
        psi_t = exact_time_evolution(H_mat, psi0_np, t)
        obs = measure_observables(psi_t, n)
        Mz_t.append(obs['Mz'])
        Mx_t.append(obs['Mx'])
    
    ax = axes[idx]
    ax.plot(times, Mz_t, 'b-', linewidth=1.5, label=r'$\langle M_z \rangle$')
    ax.plot(times, Mx_t, 'r-', linewidth=1.5, label=r'$\langle M_x \rangle$')
    ax.set_xlabel(r'Time $t$', fontsize=11)
    ax.set_ylabel('Magnetisation', fontsize=11)
    ax.set_title(f'Initial: {label}', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-1.1, 1.1)

plt.suptitle(f'Quench Dynamics under TFIM ($n={n}$, $J=h={J}$)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 10. Site-Resolved Dynamics and Information Spreading

One of the most interesting phenomena in quantum many-body dynamics is the **spreading of correlations** (the “light cone”). A local perturbation propagates at a finite velocity — the **Lieb–Robinson bound** guarantees this.

Let’s visualise how a single spin flip spreads through the chain.

In [ ]:
# Start from |10000> (single spin flip on first site) in a chain of n=6
n = 6
J, h = 1.0, 0.5
H_mat = tfim_hamiltonian(n, J, h).to_matrix()

psi0 = np.zeros(2**n, dtype=complex)
psi0[0] = 1.0  # |000000>: all up

# Flip site 0: X_0 |000000> = |100000>
X0 = SparsePauliOp.from_list([(''.join(reversed(['X'] + ['I']*(n-1))), 1.0)])
psi0_flipped = X0.to_matrix() @ psi0

# Time evolve and track <Z_i(t)>
n_times = 150
T_max = 5.0
times = np.linspace(0, T_max, n_times)

Zi_data = np.zeros((n_times, n))

for t_idx, t in enumerate(times):
    psi_t = exact_time_evolution(H_mat, psi0_flipped, t)
    obs = measure_observables(psi_t, n)
    Zi_data[t_idx, :] = obs['Zi']

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(Zi_data.T, aspect='auto', origin='lower', cmap='RdBu',
               extent=[0, T_max, -0.5, n-0.5], vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, label=r'$\langle Z_i \rangle$')
ax.set_xlabel(r'Time $t$', fontsize=12)
ax.set_ylabel(r'Site $i$', fontsize=12)
ax.set_yticks(range(n))
ax.set_title(f'Spin-flip propagation in TFIM ($n={n}$, $J={J}$, $h={h}$)', fontsize=13)
plt.tight_layout()
plt.show()

The local spin perturbation spreads through the chain over time, producing an emergent light-cone-like pattern. In local lattice systems, Lieb-Robinson bounds imply that correlations outside this effective light cone are exponentially suppressed.

In [ ]:
# Verify Trotter captures the light cone correctly
n_steps_per_t = 10  # Trotter steps per unit time

Zi_trotter = np.zeros((n_times, n))
for t_idx, t in enumerate(times):
    if t == 0:
        obs = measure_observables(Statevector(psi0_flipped), n)
        Zi_trotter[t_idx, :] = obs['Zi']
        continue
    r = max(1, int(np.ceil(n_steps_per_t * t)))
    sv_t = trotter_evolve(n, J, h, t, r, order=2, 
                          initial_state=Statevector(psi0_flipped))
    obs = measure_observables(sv_t, n)
    Zi_trotter[t_idx, :] = obs['Zi']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

im1 = ax1.imshow(Zi_data.T, aspect='auto', origin='lower', cmap='RdBu',
                 extent=[0, T_max, -0.5, n-0.5], vmin=-1, vmax=1)
plt.colorbar(im1, ax=ax1, label=r'$\langle Z_i \rangle$')
ax1.set_xlabel(r'Time $t$', fontsize=12)
ax1.set_ylabel(r'Site $i$', fontsize=12)
ax1.set_title('Exact', fontsize=13)

im2 = ax2.imshow(Zi_trotter.T, aspect='auto', origin='lower', cmap='RdBu',
                 extent=[0, T_max, -0.5, n-0.5], vmin=-1, vmax=1)
plt.colorbar(im2, ax=ax2, label=r'$\langle Z_i \rangle$')
ax2.set_xlabel(r'Time $t$', fontsize=12)
ax2.set_ylabel(r'Site $i$', fontsize=12)
ax2.set_title(f'2nd-order Trotter ($r/t = {n_steps_per_t}$)', fontsize=13)

plt.suptitle(f'Spin-flip light cone: exact vs Trotter ($n={n}$)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 11. Entanglement Entropy Growth

Another important diagnostic is the **entanglement entropy** of a bipartition. For a pure state $|\psi\rangle$ of a system split into subsystems $A$ and $B$, the von Neumann entropy of the reduced state is:

$$S(\rho_A) = -\mathrm{Tr}(\rho_A \log_2 \rho_A)$$

where $\rho_A = \mathrm{Tr}_B |\psi\rangle\langle\psi|$.

Starting from a product state, the entanglement entropy typically grows linearly in time before saturating — this is the “entanglement tsunami” picture.

In [ ]:
def entanglement_entropy(statevector, n, partition_size):
    """Compute the von Neumann entanglement entropy for a bipartition.
    
    Splits the system into [0, ..., partition_size-1] and [partition_size, ..., n-1].
    
    Parameters:
        statevector: state as a numpy array of length 2^n
        n: total number of qubits
        partition_size: number of qubits in subsystem A
    
    Returns:
        float: entanglement entropy in bits (log base 2)
    """
    psi = np.array(statevector).reshape([2] * n)
    
    # Reshape into (d_A, d_B) matrix
    d_A = 2**partition_size
    d_B = 2**(n - partition_size)
    
    # Reorder axes: A = first partition_size qubits, B = rest
    psi_matrix = psi.reshape(d_A, d_B)
    
    # SVD to get Schmidt coefficients
    singular_values = np.linalg.svd(psi_matrix, compute_uv=False)
    
    # Entanglement entropy
    p = singular_values**2
    p = p[p > 1e-12]  # Remove numerical zeros
    entropy = -np.sum(p * np.log2(p))
    
    return entropy


# Track entanglement growth
n = 6
J, h = 1.0, 1.0
H_mat = tfim_hamiltonian(n, J, h).to_matrix()

psi0 = np.zeros(2**n, dtype=complex)
psi0[0] = 1.0  # |000000>

n_times = 150
T_max = 5.0
times = np.linspace(0, T_max, n_times)

# Entropy for the half-chain bipartition
entropy_half = []
# Entropy profile (all bipartitions)
entropy_profile_early = []
entropy_profile_late = []

for t_idx, t in enumerate(times):
    psi_t = exact_time_evolution(H_mat, psi0, t)
    S = entanglement_entropy(psi_t, n, n // 2)
    entropy_half.append(S)

# Also compute the full entropy profile at a few time snapshots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(times, entropy_half, 'purple', linewidth=2)
ax1.set_xlabel(r'Time $t$', fontsize=12)
ax1.set_ylabel(r'$S(\rho_{A})$ (bits)', fontsize=12)
ax1.set_title(f'Half-chain entanglement entropy ($n={n}$, $J=h={J}$)', fontsize=13)
ax1.axhline(y=n//2, color='gray', linestyle='--', alpha=0.4, label=f'Max = {n//2} bits')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Entropy profile at different times
for t_snap in [0.5, 1.0, 2.0, 4.0]:
    psi_t = exact_time_evolution(H_mat, psi0, t_snap)
    profile = [entanglement_entropy(psi_t, n, k) for k in range(1, n)]
    ax2.plot(range(1, n), profile, 'o-', label=f'$t = {t_snap}$')

ax2.set_xlabel(r'Partition boundary (after site $k$)', fontsize=12)
ax2.set_ylabel(r'$S(\rho_{1..k})$ (bits)', fontsize=12)
ax2.set_title('Entanglement entropy profile', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The half-chain entropy grows approximately linearly at early times, then saturates near the maximum value (n/2 bits for a half-chain). This linear growth is related to the ballistic propagation of entanglement.

## 12. Resource Analysis: Circuit Depth and Gate Counts

A practical concern for near-term quantum hardware is the **circuit depth** (which determines how much noise accumulates). Let’s analyse how the Trotter circuit resources scale.

In [ ]:
print(f"{'n':>3s}  {'r':>4s}  {'Order':>5s}  {'CNOTs':>6s}  {'1q gates':>9s}  {'Depth':>6s}")
print("-" * 42)

for n in [4, 6, 8, 10]:
    for r in [5, 10]:
        for order in [1, 2]:
            qc = trotter_circuit(n, J=1.0, h=0.5, total_time=1.0, 
                                 n_steps=r, order=order)
            # Count gates
            ops = qc.count_ops()
            n_cx = ops.get('cx', 0)
            n_1q = ops.get('rx', 0) + ops.get('rz', 0)
            depth = qc.depth()
            print(f"{n:3d}  {r:4d}  {order:5d}  {n_cx:6d}  {n_1q:9d}  {depth:6d}")

print("\nFor n qubits, r steps, 1st order:")
print(f"  CNOTs per step: 2*(n-1)  |  1q gates per step: (n-1) + n = 2n-1")
print("\nFor 2nd order (half-step, full-step, half-step):")
print(f"  CNOTs per step: 2*(n-1)  |  1q gates per step: 2n + (n-1) = 3n-1")
print("\nNote: adjacent Rx gates in consecutive 2nd-order steps can be merged,")
print("reducing the actual gate count. This is left as an optimisation exercise.")

---
## 13. Exercises

### Exercise 1: Single-qubit precession

The simplest Hamiltonian simulation: a single qubit in a magnetic field.

**(a)** For $H = -\omega Z$ (Larmor precession), the time evolution is $U(t) = e^{i\omega t Z} = R_z(-2\omega t)$. Build this as a quantum circuit and verify against the exact matrix exponential for $\omega = 1$, $t = \pi/4$.

**(b)** For $H = -\omega (\sin\theta\, X + \cos\theta\, Z)$ (tilted field), build a Trotter circuit and compare with exact evolution. How many Trotter steps are needed for fidelity $> 0.99$ at $t = 5$?

**(c)** Plot $\langle Z(t) \rangle$ for both methods. The exact solution is $\langle Z(t) \rangle = \cos^2\theta + \sin^2\theta \cos(2\omega t)$. Verify numerically.

In [ ]:
# Exercise 1: Your code here


### Exercise 2: Trotter error scaling

**(a)** For the TFIM with $n = 4$, $J = 1$, $h = 0.5$, $T = 3.0$, compute the infidelity $1 - F$ as a function of Trotter steps $r$ for both first and second order.

**(b)** Fit the scaling $1 - F \sim r^{-\alpha}$ and extract the exponent $\alpha$ for each order. You should find $\alpha \approx 2$ for first order and $\alpha \approx 4$ for second order.

**(c)** Repeat for $h = 2.0$ (stronger field). Does the error increase or decrease? Why?

*Hint: The Trotter error depends on the commutator $[H_{ZZ}, H_X]$, which is proportional to $Jh$.*

In [ ]:
# Exercise 2: Your code here


### Exercise 3: Dynamics at the quantum phase transition

**(a)** Compute and plot $\langle M_z(t) \rangle$ and $\langle M_x(t) \rangle$ starting from $|00\ldots0\rangle$ for $n = 6$ and three values of $h/J$: 0.5 (ordered phase), 1.0 (critical), and 2.0 (disordered phase).

**(b)** How does the oscillation frequency and amplitude depend on $h/J$?

**(c)** Compute the long-time average $\overline{\langle M_z \rangle} = \frac{1}{T}\int_0^T \langle M_z(t)\rangle\, dt$ for $T = 20$ as a function of $h/J \in [0, 3]$. What happens near the phase transition?

In [ ]:
# Exercise 3: Your code here


### Exercise 4: Even–odd decomposition (advanced)

For an open-boundary Ising chain, the $ZZ$ bonds can be split into **even bonds** $(0,1), (2,3), \ldots$ and **odd bonds** $(1,2), (3,4), \ldots$. Within each set, the $ZZ$ terms commute (they don’t share qubits), so they can be executed **in parallel** — reducing circuit depth.

**(a)** Implement this even–odd Trotter step. For $n$ qubits, each half-step applies all even bonds simultaneously then all odd bonds, instead of applying bonds sequentially.

**(b)** Compare the circuit depth of the even–odd vs sequential implementations for $n = 8$, $r = 10$.

**(c)** Verify that the even–odd Trotter evolution gives the same fidelity as the sequential one.

*Hint: The CNOT–Rz–CNOT blocks for even (odd) bonds act on disjoint qubits and can run in the same time step.*

In [ ]:
# Exercise 4: Your code here


### Exercise 5: Periodic boundary conditions

**(a)** Modify the TFIM Hamiltonian and Trotter circuit to include **periodic boundary conditions** (PBC): add a $Z_{n-1} Z_0$ term.

**(b)** The PBC Ising chain has translation symmetry. Verify that $\langle Z_i(t) \rangle$ is the same for all sites $i$ when starting from a translationally invariant state like $|0\rangle^{\otimes n}$.

**(c)** Compare the energy spectrum with and without PBC. How does the ground-state energy change?

In [ ]:
# Exercise 5: Your code here
